# NB02: Data Transformation

**LSE ME204 – Data Engineering Principles for the Social Sciences (2026)**

**LSE ID:** 250104200

</div>

## Setup
Import packages

In [48]:
import pandas as pd
import json
import os

Function to load all json files in raw data folder for a given indicator - this will allow the analysis to be easily expanded with additional indicators.

In [49]:
def load_data(indicator):
    '''
    Load all json files within the raw data folder that matches the indicator passed to the function
    '''
    all_files = os.listdir(f"../data/raw/{indicator}")
    print(all_files)
    all_json_files = []

    for file in all_files:
            if os.path.isfile(f"../data/raw/{indicator}/{file}") and file.endswith(".json"):
                file_path = f"../data/raw/{indicator}/{file}"
                print(f"Found {file_path}")

                with open(file_path, mode="r", encoding="utf-8") as f:
                    raw_data = json.load(f)
                    # all_json_files.append(file)

                df = (pd.json_normalize(
                    raw_data["Results"]["series"], 
                    record_path="data",
                    meta="seriesID")
                )
                all_json_files.append(df)
    
    full_df = pd.concat(all_json_files)
    return full_df

raw_df = load_data("employment")

['bls_employment_4.json', 'bls_employment_3.json', 'bls_employment_2.json', 'bls_employment_1.json']
Found ../data/raw/employment/bls_employment_4.json
Found ../data/raw/employment/bls_employment_3.json
Found ../data/raw/employment/bls_employment_2.json
Found ../data/raw/employment/bls_employment_1.json


Define a function to clean the raw dataframe - allows me to repeat on other BLS series I may want to add to the analysis

In [50]:
def clean_bls(df):
    '''
    Clean the dataframe read from raw BLS response json by formatting the 
    year and period columns as datetime and dropping unneccessary columns.
    '''

    df = df.assign(
        month=df["period"].str[1:].astype(int),
        day=1
    )
    df["date"] = pd.to_datetime(df[["year","month","day"]])

    df["industry_code"] = df["seriesID"].str[3:11]

    df = df[["date", "industry_code", "value"]]
    return df

Re-map industry name and NAICS code onto the series ID using the `aiie_bls_map`.

To allow for aggregation into higher level sector groups, I add a sector column from industry codes at the 2-digit level, and match onto the supersector names in the `ce_industry` map. There are several industry names that fail to map as the BLS 2-digit level series ID does not follow the format of XX000000. I map these on manually by appending the codes of the correct format onto the mapping dataframe.

In [ ]:
clean_df = clean_bls(raw_df)

aiie_bls_map = pd.read_csv("../data/reference/aiie_bls_map.csv")
aiie_bls_map["industry_code"] = aiie_bls_map["industry_code"].astype(str)

tidy_df = pd.merge(
    left=clean_df,
    right=aiie_bls_map,
    on="industry_code",
    how="left"
)

tidy_df = tidy_df[["date", "industry_name", "industry_code", "naics", "aiie", "value"]]
tidy_df = tidy_df.rename({"aiie":"ai_exposure","value":"employment"}, axis=1)

# Load industry code map
industry_codes = pd.read_csv("../data/reference/ce_industry.tsv", delimiter="\t", index_col=False).astype(str)
# Select only sector names 
sector_names = industry_codes[["industry_code", "industry_name"]].rename(columns={"industry_name":"sector"})
# Manually add in sector names for Wholesale Trade and Utilities
additional_sectors = pd.DataFrame({
    "industry_code": ["41000000", "44000000"],
    "sector": ["Wholesale trade", "Utilities"]
})
sector_names = pd.concat([sector_names, additional_sectors])


tidy_df["supersector_code"] = tidy_df["industry_code"].astype(str).str[:2]+"000000"

tidy_df = pd.merge(left=tidy_df, right=sector_names, left_on="supersector_code", right_on="industry_code", how="left")


,industry_code,sector
0,0,Total nonfarm
1,5000000,Total private
2,6000000,Goods-producing
3,7000000,Service-providing
4,8000000,Private service-providing
...,...,...
847,90932622,Local hospitals
848,90932920,Local government general administration
849,90932999,Other local government
0,41000000,Wholesale trade


In [75]:
tidy_df.to_csv("../data/processed/employment.csv", index=False)